In [1]:
from google.colab import drive

drive.mount("/content/gdrive")

Mounted at /content/gdrive


PRIORITY-01 — Initialize

In [2]:
import os
import json
import math
import pandas as pd
import numpy as np

PROJECT_ROOT = "/content/gdrive/MyDrive/CrisisWeave_Project"

PRIORITY_RESULTS_DIR = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "Priority_Location"
)

os.makedirs(
    PRIORITY_RESULTS_DIR,
    exist_ok=True
)

print("CrisisWeave Next Priority Location module initialized.")
print("Output directory:", PRIORITY_RESULTS_DIR)

CrisisWeave Next Priority Location module initialized.
Output directory: /content/gdrive/MyDrive/CrisisWeave_Project/04_Results/Priority_Location


PRIORITY-02 — Load required outputs

In [3]:
# Load Dynamic Disaster State

state_path = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "Dynamic_State",
    "CrisisWeave_Dynamic_Disaster_State_CW001.json"
)

with open(state_path, "r", encoding="utf-8") as f:
    dynamic_state = json.load(f)


# Load Location Verification

verification_path = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "Location_Verification",
    "CrisisWeave_Location_Verification_CW001.json"
)

with open(verification_path, "r", encoding="utf-8") as f:
    location_verification = json.load(f)


# Load GIS output

gis_path = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "GIS",
    "CrisisWeave_GIS_CW001.json"
)

with open(gis_path, "r", encoding="utf-8") as f:
    gis_output = json.load(f)


print("Dynamic state loaded.")
print("Location verification loaded.")
print("GIS output loaded.")

Dynamic state loaded.
Location verification loaded.
GIS output loaded.


PRIORITY-03 — Check whether recommendation is eligible

In [4]:
verification_status = location_verification[
    "verification_status"
]

fused_severity = dynamic_state[
    "current_state"
]["severity"]

progression_direction = dynamic_state[
    "temporal_state"
]["progression_direction"]

print("Verification status :", verification_status)
print("Fused severity      :", fused_severity)
print("Progression         :", progression_direction)


priority_eligible = (
    verification_status in [
        "VERIFIED",
        "PARTIALLY VERIFIED"
    ]
    and fused_severity == "High"
)

print("\nPriority recommendation eligible:",
      priority_eligible)

Verification status : VERIFIED
Fused severity      : High
Progression         : NE

Priority recommendation eligible: True


PRIORITY-04 — Load GIS candidate locations

In [5]:
candidate_locations = gis_output.get(
    "candidate_locations",
    []
)

print("Candidate locations found:",
      len(candidate_locations))

if candidate_locations:
    print("\nFirst candidate:")
    print(json.dumps(
        candidate_locations[0],
        indent=2,
        ensure_ascii=False,
        default=str
    ))

Candidate locations found: 4

First candidate:
{
  "candidate_id": "HOSPITAL_0",
  "type": "Hospital",
  "name": "AMJ Hospital",
  "latitude": 9.9294302,
  "longitude": 76.2702904,
  "distance_from_incident_m": 38.14,
  "distance_to_water_m": 57.29,
  "distance_to_road_m": 19.62,
  "base_criticality": 1.0,
  "incident_proximity": "Very Close",
  "water_proximity": "Very Close",
  "road_proximity": "Very Close",
  "incident_proximity_score": 1.0,
  "water_proximity_score": 1.0,
  "spatial_priority": 1.0
}


PRIORITY-05 — Create candidate DataFrame

In [6]:
candidate_df = pd.DataFrame(
    candidate_locations
)

print("Candidate columns:")
print(candidate_df.columns.tolist())

print("\nNumber of candidates:",
      len(candidate_df))

display(candidate_df.head())

Candidate columns:
['candidate_id', 'type', 'name', 'latitude', 'longitude', 'distance_from_incident_m', 'distance_to_water_m', 'distance_to_road_m', 'base_criticality', 'incident_proximity', 'water_proximity', 'road_proximity', 'incident_proximity_score', 'water_proximity_score', 'spatial_priority']

Number of candidates: 4


,candidate_id,type,name,latitude,longitude,distance_from_incident_m,distance_to_water_m,distance_to_road_m,base_criticality,incident_proximity,water_proximity,road_proximity,incident_proximity_score,water_proximity_score,spatial_priority
0,HOSPITAL_0,Hospital,AMJ Hospital,9.929430,76.270290,38.14,57.29,19.62,1.00,Very Close,Very Close,Very Close,1.0,1.0,1.00
1,BRIDGE_1,Bridge,Bridge,9.930450,76.266954,409.23,9.14,0.00,0.90,Moderate,Very Close,Very Close,0.5,1.0,0.73
2,SCHOOL_0,School,Kendriya Vidyalaya-Mundamveli Kochi,9.930821,76.267321,390.06,33.99,28.47,0.75,Moderate,Very Close,Very Close,0.5,1.0,0.70
3,BRIDGE_0,Bridge,Gateway of Kochi Bridge,9.933706,76.269885,511.08,0.00,0.00,0.90,Far,Very Close,Very Close,0.2,1.0,0.58


PRIORITY-06 — Direction alignment

In [7]:
def direction_to_bearing(direction):
    direction_bearings = {
        "N": 0,
        "NE": 45,
        "E": 90,
        "SE": 135,
        "S": 180,
        "SW": 225,
        "W": 270,
        "NW": 315
    }

    return direction_bearings.get(
        direction,
        None
    )


def angular_difference(a, b):
    if a is None or b is None:
        return None

    difference = abs(a - b)

    return min(
        difference,
        360 - difference
    )


progression_bearing = direction_to_bearing(
    progression_direction
)

print("Progression direction:",
      progression_direction)

print("Progression bearing:",
      progression_bearing)

Progression direction: NE
Progression bearing: 45


PRIORITY-07 — Calculate candidate direction alignment

In [8]:
incident_lat = float(
    gis_output["incident_location"]["latitude"]
)

incident_lon = float(
    gis_output["incident_location"]["longitude"]
)


def calculate_bearing(
    lat1,
    lon1,
    lat2,
    lon2
):
    lat1 = math.radians(lat1)
    lat2 = math.radians(lat2)

    dlon = math.radians(
        lon2 - lon1
    )

    x = (
        math.sin(dlon)
        * math.cos(lat2)
    )

    y = (
        math.cos(lat1)
        * math.sin(lat2)
        -
        math.sin(lat1)
        * math.cos(lat2)
        * math.cos(dlon)
    )

    bearing = math.degrees(
        math.atan2(x, y)
    )

    return (
        bearing + 360
    ) % 360


def get_direction_alignment(
    candidate_bearing,
    progression_bearing
):
    if progression_bearing is None:
        return 0.5

    difference = angular_difference(
        candidate_bearing,
        progression_bearing
    )

    if difference <= 22.5:
        return 1.0

    if difference <= 45:
        return 0.8

    if difference <= 90:
        return 0.5

    if difference <= 135:
        return 0.2

    return 0.0


candidate_df["candidate_bearing"] = candidate_df.apply(
    lambda row: calculate_bearing(
        incident_lat,
        incident_lon,
        float(row["latitude"]),
        float(row["longitude"])
    ),
    axis=1
)

candidate_df["direction_alignment_score"] = (
    candidate_df["candidate_bearing"]
    .apply(
        lambda b: get_direction_alignment(
            b,
            progression_bearing
        )
    )
)

display(
    candidate_df[
        [
            "candidate_id",
            "type",
            "name",
            "candidate_bearing",
            "direction_alignment_score"
        ]
    ]
)

,candidate_id,type,name,candidate_bearing,direction_alignment_score
0,HOSPITAL_0,Hospital,AMJ Hospital,335.396239,0.5
1,BRIDGE_1,Bridge,Bridge,291.246843,0.2
2,SCHOOL_0,School,Kendriya Vidyalaya-Mundamveli Kochi,299.048999,0.2
3,BRIDGE_0,Bridge,Gateway of Kochi Bridge,353.251343,0.5


PRIORITY-08 — Combine spatial priority signals

In [11]:
# Make sure expected GIS score columns exist

required_columns = [
    "incident_proximity_score",
    "water_proximity_score",
    "base_criticality"
]

missing_columns = [
    col for col in required_columns
    if col not in candidate_df.columns
]

print("Missing expected columns:",
      missing_columns)


# PRIORITY-08B — Calculate final priority score

candidate_df["priority_score"] = (
    0.40 * candidate_df[
        "direction_alignment_score"
    ]
    +
    0.30 * candidate_df[
        "incident_proximity_score"
    ]
    +
    0.20 * candidate_df[
        "water_proximity_score"
    ]
    +
    0.10 * candidate_df[
        "base_criticality"
    ]
)

candidate_df["priority_score"] = (
    candidate_df["priority_score"]
    .clip(0, 1)
    .round(4)
)

candidate_df = candidate_df.sort_values(
    "priority_score",
    ascending=False
).reset_index(drop=True)

display(
    candidate_df[
        [
            "candidate_id",
            "type",
            "name",
            "distance_from_incident_m",
            "distance_to_water_m",
            "direction_alignment_score",
            "incident_proximity_score",
            "water_proximity_score",
            "base_criticality",
            "priority_score"
        ]
    ]
)

Missing expected columns: []


,candidate_id,type,name,distance_from_incident_m,distance_to_water_m,direction_alignment_score,incident_proximity_score,water_proximity_score,base_criticality,priority_score
0,HOSPITAL_0,Hospital,AMJ Hospital,38.14,57.29,0.5,1.0,1.0,1.00,0.800
1,BRIDGE_0,Bridge,Gateway of Kochi Bridge,511.08,0.00,0.5,0.2,1.0,0.90,0.550
2,BRIDGE_1,Bridge,Bridge,409.23,9.14,0.2,0.5,1.0,0.90,0.520
3,SCHOOL_0,School,Kendriya Vidyalaya-Mundamveli Kochi,390.06,33.99,0.2,0.5,1.0,0.75,0.505


PRIORITY-09 — Select next location

In [12]:
if priority_eligible and len(candidate_df) > 0:

    selected_candidate = candidate_df.iloc[0].to_dict()

    recommendation_status = (
        "PRIORITY_LOCATION_IDENTIFIED"
    )

else:

    selected_candidate = None

    recommendation_status = (
        "NO_PRIORITY_LOCATION"
    )


print("Recommendation status:",
      recommendation_status)

if selected_candidate:
    print("\nSelected next location:")
    print("Type:",
          selected_candidate["type"])
    print("Name:",
          selected_candidate["name"])
    print("Priority score:",
          selected_candidate["priority_score"])

Recommendation status: PRIORITY_LOCATION_IDENTIFIED

Selected next location:
Type: Hospital
Name: AMJ Hospital
Priority score: 0.8


PRIORITY-10 — Build recommendation object

In [13]:
if selected_candidate:

    next_priority_location = {
        "candidate_id":
            selected_candidate["candidate_id"],

        "type":
            selected_candidate["type"],

        "name":
            selected_candidate["name"],

        "latitude":
            float(selected_candidate["latitude"]),

        "longitude":
            float(selected_candidate["longitude"]),

        "distance_from_incident_m":
            float(
                selected_candidate[
                    "distance_from_incident_m"
                ]
            ),

        "priority_score":
            float(
                selected_candidate[
                    "priority_score"
                ]
            ),

        "observed_progression_direction":
            progression_direction,

        "direction_alignment_score":
            float(
                selected_candidate[
                    "direction_alignment_score"
                ]
            ),

        "reason": (
            "Candidate ranked highest based on "
            "observed temporal progression direction, "
            "spatial proximity, water proximity, "
            "and critical-location importance."
        ),

        "interpretation": (
            "This is a priority location to check next; "
            "it is not a deterministic prediction that "
            "flooding will reach this location."
        )
    }

else:

    next_priority_location = None


print(json.dumps(
    next_priority_location,
    indent=2,
    ensure_ascii=False
))

{
  "candidate_id": "HOSPITAL_0",
  "type": "Hospital",
  "name": "AMJ Hospital",
  "latitude": 9.9294302,
  "longitude": 76.2702904,
  "distance_from_incident_m": 38.14,
  "priority_score": 0.8,
  "observed_progression_direction": "NE",
  "direction_alignment_score": 0.5,
  "reason": "Candidate ranked highest based on observed temporal progression direction, spatial proximity, water proximity, and critical-location importance.",
  "interpretation": "This is a priority location to check next; it is not a deterministic prediction that flooding will reach this location."
}


PRIORITY-11 — Save ranked candidates

In [14]:
ranked_candidates_path = os.path.join(
    PRIORITY_RESULTS_DIR,
    "CrisisWeave_Ranked_Priority_Locations_CW001.csv"
)

candidate_df.to_csv(
    ranked_candidates_path,
    index=False
)

print("Ranked candidate locations saved:")
print(ranked_candidates_path)

Ranked candidate locations saved:
/content/gdrive/MyDrive/CrisisWeave_Project/04_Results/Priority_Location/CrisisWeave_Ranked_Priority_Locations_CW001.csv


PRIORITY-12 — Save final priority recommendation

In [15]:
priority_result = {
    "incident_id": dynamic_state["incident_id"],

    "eligibility": {
        "location_verification":
            verification_status,

        "fused_severity":
            fused_severity,

        "eligible":
            priority_eligible
    },

    "observed_progression": {
        "direction":
            progression_direction,

        "bearing":
            progression_bearing
    },

    "recommendation_status":
        recommendation_status,

    "next_priority_location":
        next_priority_location
}

priority_json_path = os.path.join(
    PRIORITY_RESULTS_DIR,
    "CrisisWeave_Next_Priority_Location_CW001.json"
)

with open(
    priority_json_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        priority_result,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Priority recommendation saved:")
print(priority_json_path)

Priority recommendation saved:
/content/gdrive/MyDrive/CrisisWeave_Project/04_Results/Priority_Location/CrisisWeave_Next_Priority_Location_CW001.json


PRIORITY-13 — Interactive Priority Map

In [22]:
# ============================================================
# PRIORITY-13
# FINAL INTERACTIVE NEXT PRIORITY LOCATION MAP
# ============================================================
#
# FINAL DESIGN:
#   - Esri World Street Map ONLY
#   - No layer checkbox
#   - Normal zooming
#   - No visible study-area rectangle
#   - Deep zoom uses the highest available native tile
#     and scales it instead of requesting missing tiles
#
# GIS ANALYSIS:
#   - OSM / OSMnx remains the GIS data source
#   - Basemap is only for visualization
#
# DO NOT rerun PRIORITY-01 to PRIORITY-12
# ============================================================

import os
import pandas as pd
import folium
from folium.plugins import MousePosition

# ------------------------------------------------------------
# 1. PROJECT PATH
# ------------------------------------------------------------

PROJECT_ROOT = "/content/gdrive/MyDrive/CrisisWeave_Project"

PRIORITY_DIR = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "Priority_Location"
)

RANKED_CSV = os.path.join(
    PRIORITY_DIR,
    "CrisisWeave_Ranked_Priority_Locations_CW001.csv"
)

# ------------------------------------------------------------
# 2. STUDY AREA CENTER
# ------------------------------------------------------------

CENTER_LAT = 9.9283
CENTER_LON = 76.2683

LAT_OFFSET = 0.0045
LON_OFFSET = 0.0045

NORTH = CENTER_LAT + LAT_OFFSET
SOUTH = CENTER_LAT - LAT_OFFSET
EAST = CENTER_LON + LON_OFFSET
WEST = CENTER_LON - LON_OFFSET

STUDY_BOUNDS = [
    [SOUTH, WEST],
    [NORTH, EAST]
]

# ------------------------------------------------------------
# 3. INCIDENT LOCATION
# ------------------------------------------------------------

incident_latitude = 9.929117
incident_longitude = 76.270436

# ------------------------------------------------------------
# 4. LOAD EXISTING PRIORITY RANKING
# ------------------------------------------------------------

if not os.path.exists(RANKED_CSV):

    raise FileNotFoundError(
        f"""
Ranked priority file not found:

{RANKED_CSV}

Please make sure PRIORITY-01 to PRIORITY-12
have already been completed.
"""
    )

ranked_candidates = pd.read_csv(
    RANKED_CSV
)

print("Ranked priority candidates loaded.")
print(
    "Number of candidates:",
    len(ranked_candidates)
)

# ------------------------------------------------------------
# 5. CLEAN PRIORITY SCORE
# ------------------------------------------------------------

ranked_candidates["priority_score"] = pd.to_numeric(
    ranked_candidates["priority_score"],
    errors="coerce"
)

# ------------------------------------------------------------
# 6. FIND FINAL PRIORITY LOCATION
# ------------------------------------------------------------

valid_candidates = ranked_candidates.dropna(
    subset=[
        "priority_score",
        "latitude",
        "longitude"
    ]
).copy()

if valid_candidates.empty:

    raise ValueError(
        "No valid priority candidate available."
    )

selected_candidate = valid_candidates.loc[
    valid_candidates["priority_score"].idxmax()
]

selected_name = str(
    selected_candidate.get(
        "name",
        "Unknown"
    )
)

selected_type = str(
    selected_candidate.get(
        "type",
        "Unknown"
    )
)

selected_lat = float(
    selected_candidate["latitude"]
)

selected_lon = float(
    selected_candidate["longitude"]
)

selected_score = float(
    selected_candidate["priority_score"]
)

# ------------------------------------------------------------
# 7. PRINT SELECTED LOCATION
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("NEXT PRIORITY LOCATION")
print("=" * 65)

print("Name           :", selected_name)
print("Type           :", selected_type)
print("Latitude       :", selected_lat)
print("Longitude      :", selected_lon)
print(
    "Priority Score :",
    round(selected_score, 3)
)

# ------------------------------------------------------------
# 8. CREATE MAP
# ------------------------------------------------------------
#
# IMPORTANT:
# max_zoom is high enough for normal interactive zoom.
#
# max_native_zoom=17 means:
#   - Esri tiles are requested only up to zoom 17
#   - zoom 18+ reuses/scales zoom-17 tiles
#   - therefore we avoid "Map data not yet available"
#
# The user can still zoom normally.
# ------------------------------------------------------------

priority_map = folium.Map(
    location=[
        CENTER_LAT,
        CENTER_LON
    ],
    zoom_start=16,

    # Normal interactive zoom
    min_zoom=10,
    max_zoom=22,

    # No default Folium basemap
    tiles=None,

    control_scale=True,
    zoom_control=True
)

# ------------------------------------------------------------
# 9. ESRI WORLD STREET MAP
# ------------------------------------------------------------
#
# ONLY BASEMAP.
#
# No layer control.
# No checkbox.
#
# max_native_zoom:
# Prevents requests for deeper unavailable Esri tiles.
# Leaflet scales the highest native tile instead.
# ------------------------------------------------------------

ESRI_WORLD_STREET_MAP = (
    "https://server.arcgisonline.com/"
    "ArcGIS/rest/services/"
    "World_Street_Map/"
    "MapServer/tile/{z}/{y}/{x}"
)

folium.TileLayer(
    tiles=ESRI_WORLD_STREET_MAP,

    attr=(
        "Esri, DeLorme, HERE, USGS, Intermap, "
        "iPC, NRCAN, Esri Japan, METI, Esri China "
        "(Hong Kong), Esri (Thailand), TomTom"
    ),

    name="Esri World Street Map",

    overlay=False,

    control=False,

    show=True,

    min_zoom=10,

    # Highest native level requested from Esri
    max_native_zoom=17,

    # User can continue zooming
    max_zoom=22,

    no_wrap=True
).add_to(priority_map)

# ------------------------------------------------------------
# 10. FIT INITIAL VIEW
# ------------------------------------------------------------

priority_map.fit_bounds(
    STUDY_BOUNDS,
    padding=(20, 20)
)

# ------------------------------------------------------------
# 11. KEEP MAP INSIDE STUDY AREA
# ------------------------------------------------------------
#
# IMPORTANT:
# This DOES NOT draw a rectangle.
#
# It only prevents the user from dragging
# extremely far away from the study region.
# ------------------------------------------------------------

map_name = priority_map.get_name()

max_bounds_script = f"""
<script>

document.addEventListener(
    "DOMContentLoaded",
    function() {{

        var map = {map_name};

        var southWest = L.latLng(
            {SOUTH},
            {WEST}
        );

        var northEast = L.latLng(
            {NORTH},
            {EAST}
        );

        var studyBounds = L.latLngBounds(
            southWest,
            northEast
        );

        map.setMaxBounds(
            studyBounds
        );

        map.on(
            "drag",
            function() {{

                map.panInsideBounds(
                    studyBounds,
                    {{
                        animate: false
                    }}
                );

            }}
        );

    }}
);

</script>
"""

priority_map.get_root().html.add_child(
    folium.Element(
        max_bounds_script
    )
)

# ------------------------------------------------------------
# 12. INCIDENT MARKER
# ------------------------------------------------------------

incident_popup = folium.Popup(
    """
    <div
        style="
            font-size:14px;
            line-height:1.6;
            min-width:180px;
        "
    >

        <b>CRISISWEAVE INCIDENT</b>
        <br><br>

        <b>Incident ID:</b>
        CW-001

        <br>

        <b>Severity:</b>
        HIGH

        <br>

        <b>Status:</b>
        Verified / Partially Verified

    </div>
    """,

    max_width=280
)

folium.Marker(

    location=[
        incident_latitude,
        incident_longitude
    ],

    popup=incident_popup,

    tooltip=(
        "Incident Location — CW-001"
    ),

    icon=folium.Icon(
        color="red",
        icon="warning-sign",
        prefix="glyphicon"
    )

).add_to(priority_map)

# ------------------------------------------------------------
# 13. CANDIDATE LOCATIONS
# ------------------------------------------------------------

for _, row in ranked_candidates.iterrows():

    # --------------------------------------------------------
    # Validate coordinates
    # --------------------------------------------------------

    if pd.isna(
        row.get("latitude")
    ):
        continue

    if pd.isna(
        row.get("longitude")
    ):
        continue

    lat = float(
        row["latitude"]
    )

    lon = float(
        row["longitude"]
    )

    name = str(
        row.get(
            "name",
            "Unknown"
        )
    )

    feature_type = str(
        row.get(
            "type",
            "Unknown"
        )
    )

    # --------------------------------------------------------
    # Priority score
    # --------------------------------------------------------

    score = row.get(
        "priority_score"
    )

    if pd.isna(score):

        score_text = (
            "Not calculated"
        )

    else:

        score_text = (
            f"{float(score):.3f}"
        )

    # --------------------------------------------------------
    # Incident distance
    # --------------------------------------------------------

    incident_distance = row.get(
        "distance_from_incident_m",
        None
    )

    if (
        incident_distance is None
        or pd.isna(incident_distance)
    ):

        incident_distance_text = "N/A"

    else:

        incident_distance_text = (
            f"{float(incident_distance):.2f} m"
        )

    # --------------------------------------------------------
    # Water distance
    # --------------------------------------------------------

    water_distance = row.get(
        "distance_to_water_m",
        None
    )

    if (
        water_distance is None
        or pd.isna(water_distance)
    ):

        water_distance_text = "N/A"

    else:

        water_distance_text = (
            f"{float(water_distance):.2f} m"
        )

    # --------------------------------------------------------
    # Direction alignment
    # --------------------------------------------------------

    direction_alignment = row.get(
        "direction_alignment",
        None
    )

    if (
        direction_alignment is None
        or pd.isna(direction_alignment)
    ):

        direction_text = "N/A"

    else:

        direction_text = (
            f"{float(direction_alignment):.2f}"
        )

    # --------------------------------------------------------
    # Check whether this is selected
    # --------------------------------------------------------

    is_selected = (

        abs(
            lat - selected_lat
        ) < 1e-9

        and

        abs(
            lon - selected_lon
        ) < 1e-9
    )

    # --------------------------------------------------------
    # Popup
    # --------------------------------------------------------

    popup_html = f"""
    <div
        style="
            font-size:13px;
            line-height:1.6;
            min-width:230px;
        "
    >

        <b>{name}</b>

        <br><br>

        <b>Type:</b>
        {feature_type}

        <br>

        <b>Priority Score:</b>
        {score_text}

        <br>

        <b>Distance from Incident:</b>
        {incident_distance_text}

        <br>

        <b>Distance to Water:</b>
        {water_distance_text}

        <br>

        <b>Direction Alignment:</b>
        {direction_text}

    </div>
    """

    # --------------------------------------------------------
    # Marker style
    # --------------------------------------------------------

    if is_selected:

        marker_icon = folium.Icon(
            color="green",
            icon="star",
            prefix="glyphicon"
        )

        marker_tooltip = (
            f"NEXT PRIORITY — {name}"
        )

    else:

        marker_icon = folium.Icon(
            color="blue",
            icon="info-sign",
            prefix="glyphicon"
        )

        marker_tooltip = name

    # --------------------------------------------------------
    # Add marker
    # --------------------------------------------------------

    folium.Marker(

        location=[
            lat,
            lon
        ],

        popup=folium.Popup(
            popup_html,
            max_width=320
        ),

        tooltip=marker_tooltip,

        icon=marker_icon

    ).add_to(priority_map)

# ------------------------------------------------------------
# 14. INCIDENT → PRIORITY LINE
# ------------------------------------------------------------

folium.PolyLine(

    locations=[

        [
            incident_latitude,
            incident_longitude
        ],

        [
            selected_lat,
            selected_lon
        ]

    ],

    color="blue",

    weight=4,

    opacity=0.8,

    tooltip=(
        "Incident → Next Priority Location"
    )

).add_to(priority_map)

# ------------------------------------------------------------
# 15. FINAL PRIORITY LOCATION POPUP
# ------------------------------------------------------------

selected_popup = folium.Popup(

    f"""
    <div
        style="
            font-size:14px;
            line-height:1.6;
            min-width:260px;
        "
    >

        <b>⭐ NEXT PRIORITY LOCATION</b>

        <br><br>

        <b>Name:</b>
        {selected_name}

        <br>

        <b>Type:</b>
        {selected_type}

        <br>

        <b>Priority Score:</b>
        {selected_score:.3f}

        <br><br>

        <b>Decision Basis:</b>

        <br>

        Temporal progression direction

        <br>

        Incident proximity

        <br>

        Water proximity

        <br>

        Critical infrastructure importance

    </div>
    """,

    max_width=350
)

# ------------------------------------------------------------
# 16. GREEN PRIORITY MARKER
# ------------------------------------------------------------

folium.Marker(

    location=[
        selected_lat,
        selected_lon
    ],

    popup=selected_popup,

    tooltip=(
        f"⭐ NEXT PRIORITY: "
        f"{selected_name}"
    ),

    icon=folium.Icon(
        color="green",
        icon="star",
        prefix="glyphicon"
    )

).add_to(priority_map)

# ------------------------------------------------------------
# 17. MOUSE POSITION
# ------------------------------------------------------------

MousePosition(

    position="bottomleft",

    separator=" | ",

    prefix="Lat/Lon:",

    num_digits=6

).add_to(priority_map)

# ------------------------------------------------------------
# 18. FINAL OUTPUT
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("CRISISWEAVE PRIORITY MAP READY")
print("=" * 65)

print("\nIncident:")
print(
    f"  Latitude  : {incident_latitude}"
)

print(
    f"  Longitude : {incident_longitude}"
)

print("\nNext Priority Location:")
print(
    f"  Name      : {selected_name}"
)

print(
    f"  Type      : {selected_type}"
)

print(
    f"  Latitude  : {selected_lat}"
)

print(
    f"  Longitude : {selected_lon}"
)

print(
    f"  Score     : {selected_score:.3f}"
)

print("\nBasemap:")
print(
    "  Esri World Street Map"
)

print("\nZoom:")
print(
    "  Normal interactive zoom: 10–22"
)

print(
    "  Native Esri tiles: up to zoom 17"
)

print(
    "  Deeper zoom: scaled from highest native tile"
)

print("\n" + "=" * 65)

# ------------------------------------------------------------
# 19. DISPLAY FINAL MAP
# ------------------------------------------------------------

priority_map

Ranked priority candidates loaded.
Number of candidates: 4

NEXT PRIORITY LOCATION
Name           : AMJ Hospital
Type           : Hospital
Latitude       : 9.9294302
Longitude      : 76.2702904
Priority Score : 0.8

CRISISWEAVE PRIORITY MAP READY

Incident:
  Latitude  : 9.929117
  Longitude : 76.270436

Next Priority Location:
  Name      : AMJ Hospital
  Type      : Hospital
  Latitude  : 9.9294302
  Longitude : 76.2702904
  Score     : 0.800

Basemap:
  Esri World Street Map

Zoom:
  Normal interactive zoom: 10–22
  Native Esri tiles: up to zoom 17
  Deeper zoom: scaled from highest native tile



PRIORITY-14 — Save Final Priority Map

In [23]:
# ============================================================
# PRIORITY-14: SAVE FINAL INTERACTIVE PRIORITY MAP
# ============================================================

import os

PROJECT_ROOT = "/content/gdrive/MyDrive/CrisisWeave_Project"

PRIORITY_DIR = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "Priority_Location"
)

# ------------------------------------------------------------
# Create output directory if needed
# ------------------------------------------------------------

os.makedirs(
    PRIORITY_DIR,
    exist_ok=True
)

# ------------------------------------------------------------
# Final map path
# ------------------------------------------------------------

map_path = os.path.join(
    PRIORITY_DIR,
    "CrisisWeave_Next_Priority_Map_CW001.html"
)

# ------------------------------------------------------------
# Save map
# ------------------------------------------------------------

priority_map.save(
    map_path
)

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

if os.path.exists(map_path):

    file_size_kb = (
        os.path.getsize(map_path) / 1024
    )

    print("=" * 60)
    print("PRIORITY-14 COMPLETED")
    print("=" * 60)

    print("\nFinal interactive map saved successfully.")

    print("\nPath:")
    print(map_path)

    print(
        f"\nFile size: "
        f"{file_size_kb:.2f} KB"
    )

else:

    raise FileNotFoundError(
        "Final priority map was not saved."
    )

PRIORITY-14 COMPLETED

Final interactive map saved successfully.

Path:
/content/gdrive/MyDrive/CrisisWeave_Project/04_Results/Priority_Location/CrisisWeave_Next_Priority_Map_CW001.html

File size: 17.46 KB
